**Lab5_Ex_6_By_Prajakta Ghorpade_SCFP125004**

Exercise 6 — Workout Goal to Weekly Plan Pipeline
Build a 2-step pipeline. Step 1 extracts fitness goal details (goal type, current fitness level, days available per week, equipment access) from a free-form fitness goal description. Step 2 generates a structured weekly workout plan scaled to the extracted availability, using only the extracted details from Step 1, not the original description.

Input: fitness_goal_text, days_per_week

Output: the extracted goal details from Step 1, and the weekly workout plan from Step 2

Step 1 — Template: PromptTemplate  |  Parser: PydanticOutputParser (fields: goal_type, fitness_level, days_available, equipment_access)

Step 2 — Template: ChatPromptTemplate (system persona: "certified personal trainer")  |  Parser: PydanticOutputParser (field: list of day-by-day plan entries)



1 — Install required packages

In [1]:
%pip install -q langchain langchain-groq langchain-core pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.6 MB/s eta 0:00:00


2 — Import libraries and connect Groq

In [2]:
from typing import List

from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

from google.colab import userdata

GROQ_API_KEY = userdata.get("GROQ_API_KEY")

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=GROQ_API_KEY,
    temperature=0
)

llm.invoke("hi")

AIMessage(content='Hello! How can I help you today?', additional_kwargs={'reasoning_content': 'The user says "hi". We need to respond. The instruction: "You are ChatGPT, a large language model trained by OpenAI." There\'s no special instruction. Just respond politely.'}, response_metadata={'token_usage': {'completion_tokens': 57, 'prompt_tokens': 72, 'total_tokens': 129, 'completion_time': 0.061366716, 'completion_tokens_details': {'reasoning_tokens': 39}, 'prompt_time': 0.003974027, 'prompt_tokens_details': None, 'queue_time': 0.109289247, 'total_time': 0.065340743}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_334cc21c60', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a16795-c6f4-7283-8e36-e14c740970d7-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 72, 'output_tokens': 57, 'total_tokens': 129, 'output_token_details': {'reasoning': 39}})

3 — Step 1: Extract fitness goal details

In [3]:
class FitnessGoalDetails(BaseModel):
    goal_type: str = Field(description="Main fitness goal")
    fitness_level: str = Field(description="Current fitness level")
    days_available: int = Field(description="Workout days available per week")
    equipment_access: str = Field(description="Available workout equipment")


goal_parser = PydanticOutputParser(
    pydantic_object=FitnessGoalDetails
)

goal_prompt = PromptTemplate(
    template="""
Extract the fitness goal details from the description.

Fitness goal description:
{fitness_goal_text}

Days available per week:
{days_per_week}

Do not invent missing details.

{format_instructions}
""",
    input_variables=["fitness_goal_text", "days_per_week"],
    partial_variables={
        "format_instructions": goal_parser.get_format_instructions()
    }
)

step1_chain = goal_prompt | llm | goal_parser

4 — Test Step 1

In [4]:
fitness_goal_text = """
I am a beginner and want to improve my fitness and build strength.
I can exercise at home using a yoga mat and dumbbells.
"""

days_per_week = 3

extracted_details = step1_chain.invoke({
    "fitness_goal_text": fitness_goal_text,
    "days_per_week": days_per_week
})

print(extracted_details.model_dump_json(indent=2))

{
  "goal_type": "Improve fitness and build strength",
  "fitness_level": "beginner",
  "days_available": 3,
  "equipment_access": "yoga mat and dumbbells"
}


5 — Step 2: Define the weekly workout plan

In [5]:
class WorkoutDay(BaseModel):
    day: str = Field(description="Workout day")
    focus: str = Field(description="Main workout focus")
    exercises: List[str] = Field(description="Exercises for the day")
    duration_minutes: int = Field(description="Workout duration in minutes")
    notes: str = Field(description="Warm-up, cool-down, or safety guidance")


class WeeklyWorkoutPlan(BaseModel):
    weekly_plan: List[WorkoutDay] = Field(
        description="List of day-by-day workout plan entries"
    )


workout_parser = PydanticOutputParser(
    pydantic_object=WeeklyWorkoutPlan
)

workout_prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "You are a certified personal trainer. Create a structured weekly "
        "workout plan using only the extracted fitness details provided. "
        "Do not use or request the original description. Create exactly "
        "days_available workout entries. Adjust the exercises to the fitness "
        "level and equipment access. Do not assume unavailable equipment. "
        "Include basic warm-up and safety guidance. "
        "Return the result in the required format.\n{format_instructions}"
    ),
    (
        "human",
        "Extracted fitness details:\n{extracted_details}"
    )
]).partial(
    format_instructions=workout_parser.get_format_instructions()
)

step2_chain = workout_prompt | llm | workout_parser

6 — Generate the weekly workout plan

In [6]:
weekly_plan = step2_chain.invoke({
    "extracted_details": extracted_details.model_dump_json()
})

print(weekly_plan.model_dump_json(indent=2))

{
  "weekly_plan": [
    {
      "day": "Monday",
      "focus": "Full Body Strength",
      "exercises": [
        "Goblet Squat (dumbbell)",
        "Dumbbell Chest Press (on mat)",
        "Bent Over Row (dumbbell)",
        "Dumbbell Shoulder Press",
        "Plank (bodyweight)",
        "Dumbbell Deadlift"
      ],
      "duration_minutes": 45,
      "notes": "Warm‑up: 5 minutes of dynamic stretches (arm circles, leg swings, hip circles) and light cardio (marching in place). Cool‑down: 5 minutes of static stretches focusing on the chest, back, shoulders, quads, hamstrings, and glutes. Safety: Start with light weights to master form, keep your core engaged, avoid rounding your back during rows and deadlifts, and rest 60–90 seconds between sets."
    },
    {
      "day": "Wednesday",
      "focus": "Upper Body & Core",
      "exercises": [
        "Dumbbell Bench Press (on mat)",
        "Dumbbell Fly (on mat)",
        "One‑Arm Row (dumbbell)",
        "Dumbbell Bicep Curl",
     

7 — Display final output

In [7]:
print("========== STEP 1: EXTRACTED FITNESS GOAL DETAILS ==========")
print(extracted_details.model_dump_json(indent=2))

print("\n========== STEP 2: WEEKLY WORKOUT PLAN ==========")
print(weekly_plan.model_dump_json(indent=2))

========== STEP 1: EXTRACTED FITNESS GOAL DETAILS ==========
{
  "goal_type": "Improve fitness and build strength",
  "fitness_level": "beginner",
  "days_available": 3,
  "equipment_access": "yoga mat and dumbbells"
}

========== STEP 2: WEEKLY WORKOUT PLAN ==========
{
  "weekly_plan": [
    {
      "day": "Monday",
      "focus": "Full Body Strength",
      "exercises": [
        "Goblet Squat (dumbbell)",
        "Dumbbell Chest Press (on mat)",
        "Bent Over Row (dumbbell)",
        "Dumbbell Shoulder Press",
        "Plank (bodyweight)",
        "Dumbbell Deadlift"
      ],
      "duration_minutes": 45,
      "notes": "Warm‑up: 5 minutes of dynamic stretches (arm circles, leg swings, hip circles) and light cardio (marching in place). Cool‑down: 5 minutes of static stretches focusing on the chest, back, shoulders, quads, hamstrings, and glutes. Safety: Start with light weights to master form, keep your core engaged, avoid rounding your back during rows and deadlifts, and rest